# Bond Data Cleaning & Analysis
## Bloomberg Second Batch - Mid_official, Bid_official, Ask_official

**Steps:**
1. Load data & metadata
2. Remove 26 duplicate bond pairs (keep numeric CUSIP, drop Z/B-prefixed)
3. Fix date alignment for bonds issued after 2024-03-01
4. Compute Bid-Ask Spread (bps) per CUSIP per day
5. Compute Rolling Volatility (21-day, annualized) per CUSIP per day
6. Download US Treasury curve from FRED & compute Credit Spread (G-spread) per CUSIP per day
7. Export to Excel with all tabs

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import brentq
import requests
from io import StringIO
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

FILE = 'Bloomberg_second_batch.xlsx'
OUTPUT = 'bond_analysis_output.xlsx'

# --- DUPLICATE PAIRS: keep the first (numeric CUSIP), drop the second (Z/B-prefixed) ---
# Each tuple: (KEEP, DROP)
DUPLICATE_PAIRS = [
    ('072732AC4', 'DD0134070'),
    ('05565ECH6', 'ZB0089910'),
    ('05578AAJ7', '05578BAJ5'),
    ('2027A0KK4', '2027A1KK2'),
    ('20271AAL1', '20271BAL9'),
    ('233853AM2', 'BV6694038'),
    ('24023KAF5', '24023LAF3'),
    ('36294BAL8', 'ZI1077927'),
    ('44891CCT8', '44891ACT2'),
    ('44891AAK3', '44891CAK9'),
    ('48723RAC9', '48723TAC5'),
    ('58769JAL1', 'ZI1172405'),
    ('58769JAC1', 'ZN5484896'),
    ('233851DW1', 'ZR0855455'),
    ('63859VBH3', '63859UBH5'),
    ('ZI8715974', '64953BBF4'),  # both Z-prefixed but ZI seems primary
    ('ZB0794709', '64953BBM9'),  # keeping one of each
    ('74256MEX1', '74256LEW5'),
    ('74368EBS8', '74368CBV5'),
    ('75951BAQ9', '75951AAQ1'),
    ('82620KBD4', 'BO3632995'),
    ('830505BB8', 'ZD3490319'),
    ('86563VBP3', 'ZD3710955'),
    ('86563VAY5', 'BR3356531'),
    ('86563VBA6', 'BU9466253'),
    ('86959LAL7', '86959NAL3'),
]

CUSIPS_TO_DROP = [pair[1] for pair in DUPLICATE_PAIRS]
print(f"Will drop {len(CUSIPS_TO_DROP)} duplicate CUSIPs")

## 1. Load Data

In [ ]:
# Load the three price tabs
mid_raw = pd.read_excel(FILE, sheet_name='Mid_official', header=0)
bid_raw = pd.read_excel(FILE, sheet_name='Bid_official', header=0)
ask_raw = pd.read_excel(FILE, sheet_name='Ask_official', header=0)

# Load bond metadata
meta_raw = pd.read_excel(FILE, sheet_name='Reduced_List', header=0)

# Rename first column to 'Date' for all price tabs
for df in [mid_raw, bid_raw, ask_raw]:
    df.rename(columns={df.columns[0]: 'Date'}, inplace=True)

# Drop rows where Date is NaT/None
mid_raw = mid_raw.dropna(subset=['Date']).reset_index(drop=True)
bid_raw = bid_raw.dropna(subset=['Date']).reset_index(drop=True)
ask_raw = ask_raw.dropna(subset=['Date']).reset_index(drop=True)

# Ask_official has extra rows with 0s beyond 2026-03-05 - trim to same date range as Mid
last_date = mid_raw['Date'].max()
ask_raw = ask_raw[ask_raw['Date'] <= last_date].reset_index(drop=True)
bid_raw = bid_raw[bid_raw['Date'] <= last_date].reset_index(drop=True)

# In Ask_official, 0 means missing (not an actual price of 0) - replace with NaN
ask_cols = [c for c in ask_raw.columns if c != 'Date']
ask_raw[ask_cols] = ask_raw[ask_cols].replace(0, np.nan)

# Same check for Bid
bid_cols = [c for c in bid_raw.columns if c != 'Date']
bid_raw[bid_cols] = bid_raw[bid_cols].replace(0, np.nan)

print(f"Mid: {mid_raw.shape} | Bid: {bid_raw.shape} | Ask: {ask_raw.shape}")
print(f"Date range: {mid_raw['Date'].min().date()} to {mid_raw['Date'].max().date()}")
print(f"Reduced_List: {meta_raw.shape}")
meta_raw.head(3)

## 2. Remove 26 Duplicate Bond Pairs

In [ ]:
# Drop duplicate CUSIPs from all three price DataFrames
drops_found = [c for c in CUSIPS_TO_DROP if c in mid_raw.columns]
print(f"Dropping {len(drops_found)} duplicate columns from Mid/Bid/Ask")

mid = mid_raw.drop(columns=[c for c in CUSIPS_TO_DROP if c in mid_raw.columns])
bid = bid_raw.drop(columns=[c for c in CUSIPS_TO_DROP if c in bid_raw.columns])
ask = ask_raw.drop(columns=[c for c in CUSIPS_TO_DROP if c in ask_raw.columns])

cusips = [c for c in mid.columns if c != 'Date']
print(f"Remaining bonds: {len(cusips)}")
print(f"Mid: {mid.shape} | Bid: {bid.shape} | Ask: {ask.shape}")

## 3. Fix Date Alignment for Bonds Issued After 2024-03-01

Bloomberg placed the first price in row 2 (first data row) regardless of issue date. For bonds issued after 2024-03-01, we need to shift prices down to align with the correct date.

In [ ]:
# Build CUSIP -> issue_date mapping from Reduced_List
meta_clean = meta_raw.dropna(subset=['CUSIP']).copy()
meta_clean['Issue Date'] = pd.to_datetime(meta_clean['Issue Date'])
meta_clean['Maturity'] = pd.to_datetime(meta_clean['Maturity'])
cusip_to_issue = dict(zip(meta_clean['CUSIP'], meta_clean['Issue Date']))

start_date = mid['Date'].min()
dates = mid['Date'].values

# Identify bonds that need shifting: issued after the first date in Mid_official
bonds_to_fix = []
for cusip in cusips:
    issue_dt = cusip_to_issue.get(cusip)
    if issue_dt is not None and issue_dt > start_date:
        # Find the first date in our date index >= issue date
        target_idx = np.searchsorted(dates, np.datetime64(issue_dt))
        if target_idx > 0:
            n_nans = mid[cusip].isna().sum()
            bonds_to_fix.append((cusip, issue_dt, target_idx, n_nans))

print(f"Bonds needing date realignment: {len(bonds_to_fix)}")
for cusip, idt, shift, nans in sorted(bonds_to_fix, key=lambda x: x[2]):
    print(f"  {cusip}: issued {idt.date()}, shift down by {shift} rows, current trailing NaNs: {nans}")

In [ ]:
# Apply the shift: move prices down to align with correct dates
def shift_column(df, cusip, shift_by):
    """Shift a bond's price series down by shift_by rows, filling top with NaN."""
    values = df[cusip].values.copy()
    new_values = np.full(len(values), np.nan)
    remaining = len(values) - shift_by
    if remaining > 0:
        new_values[shift_by:shift_by + remaining] = values[:remaining]
    df[cusip] = new_values

for cusip, issue_dt, shift, _ in bonds_to_fix:
    shift_column(mid, cusip, shift)
    shift_column(bid, cusip, shift)
    shift_column(ask, cusip, shift)

# Verify: check one example
if bonds_to_fix:
    ex_cusip = bonds_to_fix[0][0]
    first_valid = mid[ex_cusip].first_valid_index()
    print(f"Example: {ex_cusip} first price now at row {first_valid} = date {mid.loc[first_valid, 'Date'].date()}")
    print(f"  Issue date was: {bonds_to_fix[0][1].date()}")

print(f"\nDate alignment complete. All {len(bonds_to_fix)} bonds shifted.")

## 4. Bid-Ask Spread (basis points)

In [ ]:
# Bid-Ask Spread = (Ask - Bid) / Mid * 10,000 (in bps)
spread = pd.DataFrame({'Date': mid['Date']})

for cusip in cusips:
    ask_vals = ask[cusip] if cusip in ask.columns else np.nan
    bid_vals = bid[cusip] if cusip in bid.columns else np.nan
    mid_vals = mid[cusip]
    spread[cusip] = (ask_vals - bid_vals) / mid_vals * 10_000

print(f"Bid-Ask Spread shape: {spread.shape}")
print(f"\nSample stats (bps) across all bonds:")
spread_stats = spread[cusips].describe().loc[['mean', '50%', 'min', 'max']]
print(spread_stats.T.describe())

## 5. Rolling Volatility (21-day, annualized)

In [ ]:
# Daily returns from mid prices, then 21-day rolling std, annualized
WINDOW = 21
ANNUALIZE = np.sqrt(252)

mid_prices = mid.set_index('Date')[cusips]
daily_returns = mid_prices.pct_change()

rolling_vol = daily_returns.rolling(window=WINDOW, min_periods=WINDOW).std() * ANNUALIZE
rolling_vol = rolling_vol.reset_index()

print(f"Rolling Volatility shape: {rolling_vol.shape}")
print(f"First valid date (need {WINDOW} days of returns): {rolling_vol.dropna(subset=[cusips[0]])['Date'].min().date()}")
print(f"\nAverage annualized vol across all bonds: {rolling_vol[cusips].mean().mean():.4f} ({rolling_vol[cusips].mean().mean()*100:.2f}%)")

## 6. US Treasury Curve from FRED & Credit Spread (G-Spread)

G-Spread = Bond YTM - Interpolated Treasury yield at matching remaining maturity

In [ ]:
# Download US Treasury Constant Maturity rates from FRED
FRED_SERIES = {
    'DGS1MO': 1/12, 'DGS3MO': 3/12, 'DGS6MO': 6/12,
    'DGS1': 1, 'DGS2': 2, 'DGS3': 3, 'DGS5': 5,
    'DGS7': 7, 'DGS10': 10, 'DGS20': 20, 'DGS30': 30,
}

start = '2024-03-01'
end = '2026-03-05'

treasury_dfs = {}
for series_id, tenor in FRED_SERIES.items():
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={start}&coed={end}'
    try:
        df = pd.read_csv(url, parse_dates=['observation_date'], index_col='observation_date', na_values='.')
        df.columns = [tenor]
        treasury_dfs[tenor] = df
        print(f"  {series_id} ({tenor}y): {df.dropna().shape[0]} obs")
    except Exception as e:
        print(f"  FAILED {series_id}: {e}")

# Combine into single DataFrame
treasury = pd.concat(treasury_dfs.values(), axis=1).sort_index()
treasury = treasury.loc[start:end]

# Forward-fill missing days (weekends, holidays)
treasury = treasury.ffill()

# Treasury rates are in percent (e.g., 4.5 = 4.5%), convert to decimal for YTM comparison
treasury_decimal = treasury / 100

print(f"\nTreasury curve: {treasury.shape}")
print(f"Tenors available: {sorted(treasury.columns.tolist())}")
treasury.tail(3)

In [ ]:
# Build bond metadata lookup: coupon, frequency, maturity
cusip_meta = {}
for _, row in meta_clean.iterrows():
    c = row['CUSIP']
    if c in cusips:
        cusip_meta[c] = {
            'coupon': row['Cpn'],
            'freq': int(row['CPN_FREQ']) if pd.notna(row['CPN_FREQ']) else 2,
            'maturity': row['Maturity'],
        }

print(f"Metadata matched for {len(cusip_meta)} of {len(cusips)} bonds")

# --- Improved YTM solver: proper cash-flow schedule, BEY convention ---

def generate_coupon_dates(maturity_date, freq, valuation_date):
    """Generate future coupon dates by working backwards from maturity."""
    interval_months = 12 // freq
    dates = []
    d = pd.Timestamp(maturity_date)
    while d > pd.Timestamp(valuation_date):
        dates.append(d)
        d = d - pd.DateOffset(months=interval_months)
    dates.sort()
    return dates

def compute_ytm(price, coupon_rate, freq, maturity_date, valuation_date, face=100):
    """
    Compute YTM using actual cash flow dates and semi-annual compounding (BEY).
    This makes the result directly comparable to US Treasury yields from FRED.
    """
    if pd.isna(price) or price <= 0 or pd.isna(maturity_date):
        return np.nan
    
    mat_dt = pd.Timestamp(maturity_date)
    val_dt = pd.Timestamp(valuation_date)
    
    if mat_dt <= val_dt:
        return np.nan
    
    # Generate future coupon dates
    cpn_dates = generate_coupon_dates(mat_dt, freq, val_dt)
    if len(cpn_dates) == 0:
        return np.nan
    
    # Build cash flows: coupon at each date, plus face value at maturity
    cpn_payment = face * coupon_rate / 100 / freq
    times_yr = [(d - val_dt).days / 365.25 for d in cpn_dates]
    cfs = [cpn_payment] * len(cpn_dates)
    cfs[-1] += face  # principal returned at maturity
    
    # Solve for y (BEY): PV = sum( CF_i / (1 + y/2)^(2*t_i) ) = price
    def pv_diff(y):
        return sum(cf / (1 + y / 2) ** (2 * t) for cf, t in zip(cfs, times_yr)) - price
    
    try:
        return brentq(pv_diff, -0.05, 1.0, xtol=1e-8)
    except (ValueError, RuntimeError):
        return np.nan

# Interpolate Treasury yield for a given maturity (in years)
tenors_sorted = sorted(FRED_SERIES.values())

def interp_treasury(row, years_to_mat):
    """Linearly interpolate the Treasury curve for a given maturity."""
    if years_to_mat <= 0:
        return np.nan
    y = max(tenors_sorted[0], min(tenors_sorted[-1], years_to_mat))
    vals = [row.get(t, np.nan) for t in tenors_sorted]
    if any(np.isnan(v) for v in vals):
        # Skip NaN tenors
        valid = [(t, v) for t, v in zip(tenors_sorted, vals) if not np.isnan(v)]
        if len(valid) < 2:
            return np.nan
        ts, vs = zip(*valid)
        return np.interp(y, ts, vs)
    return np.interp(y, tenors_sorted, vals)

# Quick sanity test: a par bond should have YTM ≈ coupon rate
test_ytm = compute_ytm(100, 5.0, 2, pd.Timestamp('2029-03-01'), pd.Timestamp('2024-03-01'))
print(f"Sanity check: par bond with 5% coupon => YTM = {test_ytm*100:.3f}% (should be ~5.0%)")

test_ytm2 = compute_ytm(95, 4.0, 2, pd.Timestamp('2029-06-15'), pd.Timestamp('2024-03-01'))
print(f"Sanity check: discount bond 95, 4% cpn, 5.3yr => YTM = {test_ytm2*100:.3f}%")
print("\nYTM solver ready.")

In [ ]:
# Compute G-Spread for each bond on each date
# G-Spread = Bond YTM (BEY) - Interpolated Treasury yield (BEY) at matching remaining maturity, in bps

credit_spread = pd.DataFrame({'Date': mid['Date']})

# Reindex treasury to match our dates
treasury_reindexed = treasury_decimal.reindex(mid['Date']).ffill()

n_bonds = len(cusips)
for i, cusip in enumerate(cusips):
    if (i + 1) % 25 == 0 or i == 0:
        print(f"  Processing bond {i+1}/{n_bonds}...")
    
    meta = cusip_meta.get(cusip)
    if meta is None:
        credit_spread[cusip] = np.nan
        continue
    
    cpn = meta['coupon']
    freq = meta['freq']
    mat_date = meta['maturity']
    
    spreads = []
    for idx in range(len(mid)):
        date = mid.iloc[idx]['Date']
        price = mid.iloc[idx][cusip]
        
        if pd.isna(price) or pd.isna(mat_date):
            spreads.append(np.nan)
            continue
        
        years_to_mat = (pd.Timestamp(mat_date) - pd.Timestamp(date)).days / 365.25
        if years_to_mat <= 0:
            spreads.append(np.nan)
            continue
        
        # Bond YTM (BEY convention, using actual cash flow dates)
        ytm = compute_ytm(price, cpn, freq, mat_date, date)
        
        # Treasury yield at matching maturity
        if date in treasury_reindexed.index:
            tsy_row = treasury_reindexed.loc[date]
            tsy_yield = interp_treasury(tsy_row, years_to_mat)
        else:
            tsy_yield = np.nan
        
        if np.isnan(ytm) or np.isnan(tsy_yield):
            spreads.append(np.nan)
        else:
            spreads.append((ytm - tsy_yield) * 10_000)  # in bps
    
    credit_spread[cusip] = spreads

print(f"\nCredit Spread (G-Spread) shape: {credit_spread.shape}")

# Quick diagnostic: check for negative spreads
neg_count = (credit_spread[cusips] < 0).sum()
neg_bonds = neg_count[neg_count > 0]
print(f"\nBonds with any negative G-spread: {len(neg_bonds)}")
if len(neg_bonds) > 0:
    print("  (May indicate bonds trading tighter than Treasuries — e.g., agency/supranational debt)")
    print(neg_bonds.sort_values(ascending=False).head(10))

print(f"\nOverall mean spread: {credit_spread[cusips].mean().mean():.1f} bps")
print(f"Median spread: {credit_spread[cusips].median().median():.1f} bps")

## 6b. Credit Spread using Dirty Price (Clean + Accrued Interest)

The sawtooth pattern in G-spreads comes from using clean prices. Bloomberg quotes clean prices, but YTM should be computed from dirty price = clean price + accrued interest.

In [ ]:
# Compute accrued interest for a bond on a given date
def compute_accrued_interest(coupon_rate, freq, maturity_date, valuation_date, face=100):
    """
    Accrued interest = coupon_payment * (days since last coupon / days in coupon period).
    Uses 30/360 day count convention (standard for US corporate bonds).
    """
    mat_dt = pd.Timestamp(maturity_date)
    val_dt = pd.Timestamp(valuation_date)
    interval_months = 12 // freq
    
    # Find the last coupon date on or before valuation date
    # Work backwards from maturity
    d = mat_dt
    prev_cpn = None
    next_cpn = None
    while d > val_dt:
        next_cpn = d
        d = d - pd.DateOffset(months=interval_months)
    prev_cpn = d  # last coupon date on or before val_dt
    
    if prev_cpn is None or next_cpn is None:
        return 0.0
    
    # Days since last coupon / days in full coupon period (actual/actual)
    days_since = (val_dt - prev_cpn).days
    days_in_period = (next_cpn - prev_cpn).days
    
    if days_in_period <= 0:
        return 0.0
    
    coupon_payment = face * coupon_rate / 100 / freq
    return coupon_payment * (days_since / days_in_period)

# Sanity test
ai_test = compute_accrued_interest(5.0, 2, '2029-03-01', '2024-06-01')
print(f"Sanity check: 5% semi-annual bond, 3 months after coupon => AI = {ai_test:.4f} (expect ~1.25)")

# Compute G-Spread using dirty price = clean price + accrued interest
credit_spread_dirty = pd.DataFrame({'Date': mid['Date']})

n_bonds = len(cusips)
for i, cusip in enumerate(cusips):
    if (i + 1) % 25 == 0 or i == 0:
        print(f"  Processing bond {i+1}/{n_bonds}...")
    
    meta = cusip_meta.get(cusip)
    if meta is None:
        credit_spread_dirty[cusip] = np.nan
        continue
    
    cpn = meta['coupon']
    freq = meta['freq']
    mat_date = meta['maturity']
    
    spreads = []
    for idx in range(len(mid)):
        date = mid.iloc[idx]['Date']
        clean_price = mid.iloc[idx][cusip]
        
        if pd.isna(clean_price) or pd.isna(mat_date):
            spreads.append(np.nan)
            continue
        
        years_to_mat = (pd.Timestamp(mat_date) - pd.Timestamp(date)).days / 365.25
        if years_to_mat <= 0:
            spreads.append(np.nan)
            continue
        
        # Dirty price = clean price + accrued interest
        ai = compute_accrued_interest(cpn, freq, mat_date, date)
        dirty_price = clean_price + ai
        
        # Bond YTM from dirty price
        ytm = compute_ytm(dirty_price, cpn, freq, mat_date, date)
        
        # Treasury yield at matching maturity
        if date in treasury_reindexed.index:
            tsy_row = treasury_reindexed.loc[date]
            tsy_yield = interp_treasury(tsy_row, years_to_mat)
        else:
            tsy_yield = np.nan
        
        if np.isnan(ytm) or np.isnan(tsy_yield):
            spreads.append(np.nan)
        else:
            spreads.append((ytm - tsy_yield) * 10_000)  # in bps
    
    credit_spread_dirty[cusip] = spreads

print(f"\nCredit Spread (Dirty Price) shape: {credit_spread_dirty.shape}")

# Compare with clean price version
neg_clean = (credit_spread[cusips] < 0).sum().sum()
neg_dirty = (credit_spread_dirty[cusips] < 0).sum().sum()
print(f"Negative spread observations: Clean={neg_clean}, Dirty={neg_dirty}")
print(f"Mean spread: Clean={credit_spread[cusips].mean().mean():.1f} bps, Dirty={credit_spread_dirty[cusips].mean().mean():.1f} bps")

## 7. Export to Excel

In [ ]:
# Export everything to a single Excel file with multiple tabs
with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    mid.to_excel(writer, sheet_name='Mid_Clean', index=False)
    bid.to_excel(writer, sheet_name='Bid_Clean', index=False)
    ask.to_excel(writer, sheet_name='Ask_Clean', index=False)
    spread.to_excel(writer, sheet_name='BidAsk_Spread_bps', index=False)
    rolling_vol.to_excel(writer, sheet_name='Rolling_Vol_21d', index=False)
    credit_spread.to_excel(writer, sheet_name='Credit_Spread_bps', index=False)
    credit_spread_dirty.to_excel(writer, sheet_name='Credit_Spread_Dirty', index=False)
    
    # Also save the Treasury curve for reference
    treasury.reset_index().to_excel(writer, sheet_name='Treasury_Curve', index=False)

print(f"Exported to {OUTPUT}")
print(f"\nSheets:")
print(f"  Mid_Clean            - Cleaned mid prices ({mid.shape})")
print(f"  Bid_Clean            - Cleaned bid prices ({bid.shape})")
print(f"  Ask_Clean            - Cleaned ask prices ({ask.shape})")
print(f"  BidAsk_Spread_bps    - Daily bid-ask spread in bps ({spread.shape})")
print(f"  Rolling_Vol_21d      - 21-day rolling annualized vol ({rolling_vol.shape})")
print(f"  Credit_Spread_bps    - G-Spread from clean price in bps ({credit_spread.shape})")
print(f"  Credit_Spread_Dirty  - G-Spread from dirty price in bps (no sawtooth) ({credit_spread_dirty.shape})")
print(f"  Treasury_Curve       - FRED Treasury rates used ({treasury.shape})")
print(f"\nAll tabs: {len(cusips)} bonds x {len(mid)} trading days")